# Exercise 4.3.15 — measure suppression effect with KL divergence

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `4.3 Interpreting Reasoning Models`  
**Notebook:** `4.3_Interpreting_Reasoning_Models_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=4.3.15](https://delta-drills.vercel.app/?arena_exercise=4.3.15)


# [4.3] Interpreting Reasoning: Thought Anchors (exercises)

> **ARENA [Streamlit Page](https://arena-chapter4-alignment-science.streamlit.app/03_[4.3]_Interpreting_Reasoning_Models)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter4_alignment_science/exercises/part3_interpreting_reasoning_models/4.3_Interpreting_Reasoning_Models_exercises.ipynb?t=20260329) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter4_alignment_science/exercises/part3_interpreting_reasoning_models/4.3_Interpreting_Reasoning_Models_solutions.ipynb?t=20260329)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/header-64.png" width="350">

# Introduction

So far in this chapter, we've focused on dividing LLM computation into small steps: single-token generation. We saw this in Indirect Object Identification where we analyzed how GPT2-Small generates a single token. But modern reasoning models produce very long **chain-of-thought** traces, so we need to think about serialized computation over many tokens, not just over layers. This requires a new abstraction.

The [Thought Anchors](https://arxiv.org/abs/2506.19143) paper introduces this abstraction by splitting reasoning traces by **sentence**. Sentences are more coherent than tokens and correspond more closely to the actual reasoning steps. Some sentences matter a lot more than others for shaping the reasoning trajectory and the final answer; the authors call these **thought anchors**. They can be identified using black-box methods (resampling rollouts) or white-box methods (looking at / intervening on attention patterns). In these exercises, we'll work through both.

<img src="https://i.snipboard.io/PBoc9G.jpg" width="700">

We'll also explore the [Thought Branches paper](https://arxiv.org/abs/2510.27484) which extends these techniques to safety-relevant scenarios like blackmail, introducing metrics like **resilience** to distinguish genuine causal drivers from post-hoc rationalizations.

## Content & Learning Objectives

### 1️⃣ CoT Infrastructure & Sentence Taxonomy

> ##### Learning Objectives
>
> - Load and explore the MATH reasoning dataset structure (prompts, CoT traces, answers)
> - Implement sentence segmentation for chain-of-thought traces using regex patterns
> - Build both rule-based and LLM-based classifiers to categorize reasoning steps (problem_setup, active_computation, fact_retrieval, etc.)
> - Visualize how different sentence types are distributed across reasoning traces

### 2️⃣ Black-box Analysis

> ##### Learning Objectives
>
> - Understand **forced answer importance**: measuring impact by directly forcing model answers at specific steps
> - Implement **resampling importance**: measuring impact by having the model regenerate its own continuation from that point forward
> - Implement **counterfactual importance**: measuring impact by resampling then filtering to steps where the regenerated sentence is semantically different from the original
> - Learn when each metric is appropriate (causal vs correlational analysis)
> - Replicate key paper figures showing which reasoning steps are most critical for final answers

### 3️⃣ White-box Methods

> ##### Learning Objectives
>
> - Understand **receiver heads**: attention heads that aggregate information from reasoning steps into the final answer
> - Compute **vertical attention scores**: quantifying how much each token position attends to specific reasoning steps
> - Implement RoPE (Rotary Position Embeddings) to understand positional encoding in attention
> - Build attention suppression interventions to causally validate which attention patterns matter
> - Compare white-box attention metrics with black-box importance scores to validate mechanistic understanding

### 4️⃣ Thought Branches: Safety Applications

> ##### Learning Objectives
>
> - Apply thought anchor analysis to safety-critical scenarios (blackmail reasoning)
> - Measure how much reasoning steps influence safety-relevant outcomes (not just answer correctness)
> - Understand the **resilience metric**: how many times a sentence must be iteratively removed before it stays absent from the model's regeneration
> - Compare thought anchor patterns between mathematical reasoning (MATH benchmark) and safety reasoning (blackmail scenarios)

## Reading Material

Read at least the introduction and methods of the Thought Anchors paper before starting - the exercises follow its methodology closely. The Thought Branches paper is needed for Section 4.

- [Thought Anchors: Which LLM Reasoning Steps Matter?](https://arxiv.org/abs/2506.19143). Introduces the abstraction of treating sentences (not tokens) as units of reasoning in chain-of-thought traces, and identifies "thought anchors" - the sentences that most influence the final answer. Sections 1-3 of the exercises replicate the paper's black-box and white-box methods. Read the abstract, Section 1 (introduction) as well as 2 (explaining the methods for quantifying sentence importance) and 3 (explaining the sentence taxonomy). There's also a nice [interactive demo](https://thought-anchors.com/) showing thought anchors on example traces.
- [Thought Branches: Interpreting LLM Reasoning Requires Resampling](https://arxiv.org/abs/2510.27484). Extends thought anchor analysis to safety-relevant scenarios (e.g. blackmail reasoning), introducing the "resilience" metric. Mostly we just recommend skimming sections 1-3.
- [Principled Interpretability of Reward Hacking in Closed Frontier Models](https://www.lesswrong.com/posts/A67SbpTjuXEHK8Cvo/principled-interpretability-of-reward-hacking-in-closed) by Kroiz, Singh, Rajamanoharan & Nanda (MATS 9.0). Adapts the thought anchors resampling methodology to study reward hacking in closed frontier models (GPT-5, o3, Gemini 3 Pro), using agent actions as units of analysis rather than CoT sentences. Not needed for the exercises, but gives a sense of where the methodology is being applied.

## Setup code

Before running this code, you'll need to clone the [thought-anchors repo](https://github.com/interp-reasoning/thought-anchors). Make sure you're cloning it inside the `chapter4_alignment_science/exercises` directory.

```bash
cd chapter4_alignment_science/exercises
git clone https://github.com/interp-reasoning/thought-anchors.git
```

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter4_alignment_science"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import transformer_lens
except:
    %pip install transformer_lens==2.17.0 einops jaxtyping openai

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else repo
    if Path(repo).exists()
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import json
import math
import os
import re
import sys
import time
import warnings
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from types import MethodType
from typing import Any, Callable

import circuitsvis as cv
import einops
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import torch as t
import transformers
from datasets import load_dataset
from dotenv import load_dotenv
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import disable_progress_bars, enable_progress_bars
from IPython.display import HTML, display
from jaxtyping import Float, Int
from openai import OpenAI
from plotly.subplots import make_subplots
from scipy import stats
from sentence_transformers import SentenceTransformer
from torch import Tensor
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria

warnings.filterwarnings("ignore")

thought_anchors_path = Path.cwd() / "thought-anchors"
assert thought_anchors_path.exists(), f"Please clone thought-anchors repo as {thought_anchors_path.resolve()!r}"

sys.path.append(str(thought_anchors_path))

t.set_grad_enabled(False)

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")
dtype = t.bfloat16

# Make sure exercises are in the path
chapter = "chapter4_alignment_science"
section = "part3_interpreting_reasoning_models"
repo = "ARENA_3.0"
root_dir = next(p for p in Path.cwd().parents if p.name == repo)
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

import part3_interpreting_reasoning_models.tests as tests
import part3_interpreting_reasoning_models.utils as utils

MAIN = __name__ == "__main__"

You'll need to create an `.env` file in the `chapter4_alignment_science/exercises` directory before running the next cell.

In [ ]:
# Load .env file
env_path = exercises_dir / ".env"
assert env_path.exists(), "Please create a .env file with your API keys"

load_dotenv(dotenv_path=str(env_path))

# Setup OpenRouter client (works with both Claude and OpenAI models)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
assert OPENROUTER_API_KEY, "Please set OPENROUTER_API_KEY in your .env file"

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

Now let's define some constants:

In [ ]:
print(f"Using device: {device}")

# 1️⃣ CoT Infrastructure & Sentence Taxonomy

> ##### Learning Objectives
>
> - Load and explore the MATH reasoning dataset structure (prompts, CoT traces, answers)
> - Implement sentence segmentation for chain-of-thought traces using regex patterns
> - Build both rule-based and LLM-based classifiers to categorize reasoning steps (problem_setup, active_computation, fact_retrieval, etc.)
> - Visualize how different sentence types are distributed across reasoning traces

In this section, we'll build infrastructure to analyze chain-of-thought reasoning by breaking it into meaningful units. When models generate long reasoning traces, we need to decide what the right "units of reasoning" are.

Tokens are too granular: a single reasoning step like "Let's calculate the area: $\pi r^2$" spans multiple tokens, and splitting it apart loses semantic meaning. Sentences are a more natural unit. They correspond to complete thoughts, they're granular enough to identify specific important steps, and they align with how humans chunk reasoning.

By splitting CoT traces into sentences and categorizing them (calculations, lookups, restatements, etc.), we can ask: which types are most critical for reaching the correct answer? This sentence-level analysis sets us up for the black-box and white-box methods in later sections.

## Model Setup & Dataset Inspection

We'll start by setting a few constants (the purpose of these will become clear as we go through the exercises):

In [ ]:
# Configuration
MODEL_NAME_1B = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MODEL_NAME_8B = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
MODEL_NAME_14B = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
DATASET_NAME = "uzaymacar/math-rollouts"
BLACKMAIL_DATASET_NAME = "uzaymacar/blackmail-rollouts"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
SIMILARITY_THRESHOLD = 0.8

Next, we'll load an embedding model. Embedding models take in text and output a vector - we'll use them to measure similarity between sentences, so we can find motifs in our reasoning traces.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print(embedding_model)

To understand how embedding models work, let's look at cosine similarities between a few example sentences:

In [ ]:
prompts = [
    "Wait, I think I made an error in my reasoning and need to backtrack",
    "Hold on, I believe I made a mistake in my logic and should reconsider",
    "After careful analysis, I've determined the correct answer is 42",
    "Time is an illusion. Lunchtime doubly so.",
]
labels = [x[:35] + "..." for x in prompts]

embedding = embedding_model.encode(prompts)
cosine_sims = embedding @ embedding.T

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cosine_sims, cmap="RdBu", vmin=-1, vmax=1)
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(labels, fontsize=9)
plt.colorbar(im, label="Cosine Similarity")
plt.title("Sentence Embedding Similarity")
plt.tight_layout()
plt.show()

We can also load the dataset that the paper's authors open-sourced. The dataset is very large, but the authors provide the structure on the [HuggingFace page](https://huggingface.co/datasets/uzaymacar/math-rollouts), so we can use `huggingface_hub` to load just the data we want. We'll inspect it more shortly.

In [ ]:
PROBLEM_ID = 4682

path = f"deepseek-r1-distill-llama-8b/temperature_0.6_top_p_0.95/correct_base_solution/problem_{PROBLEM_ID}/base_solution.json"
local_path = hf_hub_download(repo_id=DATASET_NAME, filename=path, repo_type="dataset")

with open(local_path, "r") as f:
    problem_data = json.load(f)

print("Keys in problem data:", list(problem_data.keys()))
print(f"\nProblem prompt:\n{problem_data['prompt']}")

## Sentence Splitting

First, we'll need to split our CoT traces into sentences based on punctuation and paragraph breaks. We'll also need to handle special tokens like `<think>`. We've provided the function `split_solution_into_chunks` below, which implements the following rules:

- The `<think> ... </think>` tags should be removed
- You should split on sentences (i.e. ending in any of `.`, `!`, `?`, or newlines), and characters like `:`
- You should split on periods `.` unless they are decimal numbers e.g. `x.y` or a numbered list e.g. `\n1.`
- No chunk should have length less than 10: if so, then merge it with the next chunk
- Each chunk should be stripped of whitespace

Read through the function and run the test cases below to verify it works on basic inputs.

### Recommended reading: edge cases in sentence splitting

After running the test cases, try to come up with your own input where this function breaks down or splits into chunks in a counter-intuitive way. Some things to consider: what happens with URLs, abbreviations (e.g. "Dr. Smith"), LaTeX expressions (e.g. `$3.14$`), or ellipses (`...`)? Once you've found an edge case, see if you can modify the function to handle it.

In [ ]:
def split_solution_into_chunks(text: str) -> list[str]:
    """Split solution into sentence-level chunks."""
    # Remove thinking tags
    if "<think>" in text:
        text = text.split("<think>")[1]
    if "</think>" in text:
        text = text.split("</think>")[0]
    text = text.strip()

    # Replace "." characters which we don't want to split on
    text = re.sub(r"(\d)\.(\d)", r"\1<DECIMAL>\2", text)  # e.g. "4.5" -> "4<DECIMAL>5"
    text = re.sub(r"\n(\d)\.(\s)", r"\n\1<DECIMAL>\2", text)  # e.g. "\n1. " -> "\n1<DECIMAL> "

    # Split on sentence endings, combining endings with previous chunk
    sentences = re.split(r"([!?:\n]|(?<!\n\d)\.)", text)
    chunks = []
    for i in range(0, len(sentences) - 1, 2):
        chunks.append((sentences[i] + sentences[i + 1]).replace("\n", " "))

    # Replace <DECIMAL> back with "."
    chunks = [re.sub(r"<DECIMAL>", ".", c) for c in chunks]

    # Merge chunks that are too short
    if not chunks:
        return []
    merged = [chunks[0]]
    for c in chunks[1:]:
        if len(merged[-1]) < 10:
            merged[-1] += c
        else:
            merged.append(c)
    return [c.strip() for c in merged if c.strip()]


test_cases = [
    (
        "<think>First, I understand the problem. Next, I'll solve for x. Finally, I verify!</think>",
        ["First, I understand the problem.", "Next, I'll solve for x.", "Finally, I verify!"],
    ),
    (
        "<think>Let me break this down: 1. Convert to decimal. 2. Calculate log. 3. Apply formula.</think>",
        ["Let me break this down:", "1. Convert to decimal.", "2. Calculate log.", "3. Apply formula."],
    ),
    (
        "<think>The answer is 42. Done.</think>",
        ["The answer is 42.", "Done."],
    ),
]

for input_text, expected_chunks in test_cases:
    chunks = split_solution_into_chunks(input_text)
    assert chunks == expected_chunks, f"Expected {expected_chunks}, got {chunks}"

print("All tests passed!")

## Sentence Categorization

The paper uses a taxonomy of 8 categories: Problem Setup (parsing/rephrasing the problem), Plan Generation (stating a plan of action), Fact Retrieval (recalling facts, formulas, problem details), Active Computation (algebra, calculations, manipulations), Uncertainty Management (expressing confusion, re-evaluating, backtracking), Result Consolidation (aggregating intermediate results), Self Checking (verifying previous steps), and Final Answer Emission (explicitly stating the final answer).

We define these below:

In [ ]:
CATEGORIES = {
    "problem_setup": "Problem Setup",
    "plan_generation": "Plan Generation",
    "fact_retrieval": "Fact Retrieval",
    "active_computation": "Active Computation",
    "uncertainty_management": "Uncertainty Management",
    "result_consolidation": "Result Consolidation",
    "self_checking": "Self Checking",
    "final_answer_emission": "Final Answer Emission",
    "unknown": "Unknown",
}

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "4.3.15"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part3_interpreting_reasoning_models.solutions import categorize_sentences_heuristic, generate_response, calculate_answer_importance, calculate_counterfactual_importance, precompute_rollout_embeddings, precompute_rollout_embeddings, get_resampled_rollouts, extract_attention_matrix, get_vertical_scores_simple, get_vertical_scores, find_receiver_heads, compute_receiver_head_scores, rotate_half, compute_suppressed_attention


### Exercise - measure suppression effect with KL divergence

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵⚪⚪
> 
> You should spend up to 15-20 minutes on this exercise.
> ```

For a real model, we'd measure how suppressing attention to each sentence changes the output distribution.

In [ ]:
def compute_suppression_kl(
    original_logits: Float[np.ndarray, " vocab"],
    suppressed_logits: Float[np.ndarray, " vocab"],
    temperature: float = 1.0,
) -> float:
    """
    Compute KL divergence between original and suppressed output distributions.

    Args:
        original_logits: Logits from original forward pass
        suppressed_logits: Logits from suppressed forward pass
        temperature: Temperature for softmax

    Returns:
        KL divergence (in nats)
    """

    raise NotImplementedError("Compute KL divergence between distributions")


tests.test_compute_suppression_kl(compute_suppression_kl)

<details><summary>Solution</summary>

```python
def compute_suppression_kl(
    original_logits: Float[np.ndarray, " vocab"],
    suppressed_logits: Float[np.ndarray, " vocab"],
    temperature: float = 1.0,
) -> float:
    """
    Compute KL divergence between original and suppressed output distributions.

    Args:
        original_logits: Logits from original forward pass
        suppressed_logits: Logits from suppressed forward pass
        temperature: Temperature for softmax

    Returns:
        KL divergence (in nats)
    """

    def softmax(x, temp):
        x = x / temp
        exp_x = np.exp(x - np.max(x))
        return exp_x / exp_x.sum()

    p = softmax(original_logits, temperature)
    q = softmax(suppressed_logits, temperature)

    # Add small epsilon for numerical stability
    eps = 1e-10
    return np.sum(p * np.log((p + eps) / (q + eps)))
```
</details>

### Testing attention suppression

Now let's test attention suppression using forward method replacement. The key point is that we mask attention scores BEFORE softmax, not after - the implementation above replaces the entire forward method to intervene at the right point. We expect that suppressing high-importance sentences causes larger KL divergence than suppressing low-importance sentences.

In [ ]:
# Select high and low importance sentences
valid_mask = ~np.isnan(receiver_scores)
valid_indices = np.where(valid_mask)[0]
valid_scores = receiver_scores[valid_mask]

high_idx = valid_indices[np.argmax(valid_scores)]
low_idx = valid_indices[np.argmin(valid_scores)]

print(f"\nHigh importance sentence {high_idx}: score={receiver_scores[high_idx]:.4f}")
print(f"  Text: {sentences_subset[high_idx][:100]}...")
print(f"\nLow importance sentence {low_idx}: score={receiver_scores[low_idx]:.4f}")
print(f"  Text: {sentences_subset[low_idx][:100]}...")

# Get token boundaries
high_sent_range = boundaries[high_idx]
low_sent_range = boundaries[low_idx]

# Prepare input
inputs = tokenizer(text_subset, return_tensors="pt")
if model.device.type == "cuda":
    inputs = {k: v.cuda() for k, v in inputs.items()}

# Original forward pass (no suppression)
with t.no_grad():
    original_output = model(**inputs)
    original_logits = original_output.logits[0, -1].cpu().numpy()

layer_to_heads = {}
for layer_idx, head_idx in receiver_heads[:5]:
    if layer_idx not in layer_to_heads:
        layer_to_heads[layer_idx] = []
    layer_to_heads[layer_idx].append(int(head_idx))

# Test 1: Suppress high-importance sentence in top receiver heads
suppression_info = apply_qwen_attention_suppression(model, high_sent_range, layer_to_heads)

with t.no_grad():
    suppressed_high_output = model(**inputs)
    suppressed_high_logits = suppressed_high_output.logits[0, -1].cpu().numpy()

remove_qwen_attention_suppression(model, suppression_info)

kl_high = compute_suppression_kl(original_logits, suppressed_high_logits)

# Test 2: Suppress low-importance sentence
suppression_info = apply_qwen_attention_suppression(model, low_sent_range, layer_to_heads)

with t.no_grad():
    suppressed_low_output = model(**inputs)
    suppressed_low_logits = suppressed_low_output.logits[0, -1].cpu().numpy()

remove_qwen_attention_suppression(model, suppression_info)

kl_low = compute_suppression_kl(original_logits, suppressed_low_logits)

# Compare results
print(f"\nKL divergence when suppressing HIGH importance sentence: {kl_high:.4e}")
print(f"KL divergence when suppressing LOW importance sentence:  {kl_low:.4e}")
print(f"Ratio (high/low): {kl_high / (kl_low + 1e-10):.2f}x")

if kl_high > kl_low * 10:
    print("\nSuccess! Suppressing high-importance sentences causes larger output changes")
    print("  This causally validates that receiver head scores identify important sentences")

What this shows: suppressing attention to high-importance sentences (as identified by receiver head scores) causes much larger KL divergence than suppressing low-importance ones. This is a causal validation - it confirms that the receiver head scores aren't just correlational, but that attention to these sentences actually drives the model's output.

Note: the suppression works by replacing attention module forward methods (not post-hooks) and masking attention scores to -inf before softmax. This is specific to the Qwen/Qwen2 architecture.

### Bonus Exercises: Replicating Paper Whitebox Results

The correlation analyses above produce weak results on a single reasoning trace - this is expected. Robust claims about thought anchors require aggregating evidence across many problems. The paper's whitebox experiments run over a large dataset of math rollouts (available on [HuggingFace](https://huggingface.co/datasets/uzaymacar/math-rollouts)) and replicate across two model families (Qwen-14B and Llama-8B).

The bonus exercises below each target a specific figure from the paper. Each has a starting point using the infrastructure already built in this notebook, and a stretch goal that scales up to the full dataset and the paper's code from the `whitebox-analyses/` directory.

### Replicate the sentence-to-sentence suppression KL matrix (Figure 6)

The paper's most direct causal whitebox result is a sentence-to-sentence suppression matrix: for each pair of sentences $(j, i)$, suppress attention to sentence $j$ across all attention heads, then measure the mean KL divergence of the model's token-level predictions within sentence $i$. This produces an $N \times N$ matrix where entry $[i, j]$ captures how much sentence $j$ causally influences predictions at positions in sentence $i$. This is the whitebox complement of the blackbox counterfactual importance computed earlier, and Figure 6 visualizes it as a heatmap.

Starting point (single problem): using the model and suppression tools from this notebook:

1. Run a baseline forward pass and collect **all** token-level logits (not just the final position) - you will need the full logit tensor, not just the last position slice.
2. For each sentence $j$, call `apply_qwen_attention_suppression` across **all** attention heads (set `layer_to_heads` to include every head in every layer), run a forward pass to collect token-level logits, then restore with `remove_qwen_attention_suppression`. This is compute-intensive - start with a subset of sentences.
3. For each pair $(j, i)$, compute the mean KL divergence between baseline and suppressed logits over all token positions that fall within sentence $i$'s token boundaries (use `compute_suppression_kl` from earlier in the notebook). Store results in an $N \times N$ matrix.
4. Mask the upper triangle and diagonal with `np.nan` (future sentences cannot be causally affected by later ones), then visualize as a heatmap. Try subtracting each row's mean to normalize for sentence depth. Use a white-to-red colormap.

What to look for: thought anchors appear as columns with elevated KL values stretching down across many rows - they causally affect many later sentences. These columns typically correspond to plan generation or explicit backtracking steps. Compare the column positions to the receiver head scores and blackbox counterfactual importances from Section 2.

Stretch goal (full dataset): the paper's implementation is `get_suppression_KL_matrix` in `whitebox-analyses/attention_analysis/attn_supp_funcs.py`. It uses nucleus-sampling logit compression (`p_nucleus=0.9999`) to store only the top-p logits efficiently. Run it on 10-20 problems and average the row-normalized matrices. The `plot_suppression_matrix.py` script handles visualization. Compare your result to Figure 6.

In [ ]:
def replicate_figure_6(
    model,
    tokenizer,
    input_ids: Tensor,
    sentence_boundaries: list[tuple[int, int]],
    layer_to_heads: dict[int, list[int]] | None = None,
    row_normalize: bool = True,
) -> np.ndarray:
    """
    Compute the sentence-to-sentence suppression KL matrix,
    replicating Figure 6 from the Thought Anchors paper.

    For a single reasoning trace with N sentences:
    1. Run a baseline forward pass and collect all token-level logits
       (the full logit tensor, not just the last position)
    2. For each sentence j (the suppressed sentence), call
       apply_qwen_attention_suppression to suppress attention to
       sentence j's tokens across all heads, run a forward pass to
       get suppressed logits, then restore with
       remove_qwen_attention_suppression
    3. For each pair (j, i), compute mean KL divergence between
       baseline and suppressed logits over all token positions within
       sentence i's boundaries, using compute_suppression_kl
    4. Store results in an N x N matrix where entry [i, j] is the
       causal influence of sentence j on predictions at sentence i
    5. Mask the upper triangle and diagonal with np.nan (future
       sentences cannot be causally affected by later ones)
    6. Optionally subtract each row's mean to normalize for sentence
       depth effects

    Returns the N x N suppression KL matrix (lower-triangular, with
    np.nan on and above the diagonal).
    """
    raise NotImplementedError()

### Identify receiver heads via kurtosis and verify split-half reliability (Figure 4)

The paper identifies "receiver heads" by computing the kurtosis of each attention head's vertical attention scores across a set of reasoning traces. The vertical score for sentence $s$ under head $(l, h)$ is the mean attention weight that $s$ receives from all subsequent token positions, after ignoring a local proximity window. High kurtosis means a head's attention is concentrated on a small number of "anchor" sentences rather than spread broadly. The paper finds that only a small fraction of heads have high kurtosis, these heads are consistent across problems, and they correspond to the receiver heads identified qualitatively earlier.

Starting point (single problem): using the current reasoning trace and loaded model:

1. For each attention head $(l, h)$, extract the full attention matrix and compute sentence-level vertical scores: for each sentence $s$, average the attention weights directed *toward* sentence $s$ from all token positions that are at least `proximity_ignore=16` positions after sentence $s$'s end boundary. The paper implements this in `get_vertical_scores` in `whitebox-analyses/attention_analysis/attn_funcs.py`.
2. Compute `scipy.stats.kurtosis` of the resulting per-sentence score array for each head.
3. Produce a scatter plot of kurtosis versus layer index. You should see a small number of outlier heads with substantially higher kurtosis than the bulk of heads (which cluster near zero). These are candidate receiver heads.
4. Check whether the top-kurtosis heads from this single trace agree with the receiver head set used in this notebook (which was pre-computed from many traces).

Stretch goal - split-half reliability (paper result: r = 0.84): replicate the paper's reliability analysis across many reasoning traces:

1. Download the dataset from HuggingFace (`uzaymacar/math-rollouts`) and process a set of problems using `get_all_problems_vert_scores` from the paper's `receiver_head_funcs.py`.
2. Split problems into two halves (odd/even indices). For each half, average per-head kurtosis using `get_3d_ar_kurtosis`.
3. Scatter plot first-half vs. second-half mean kurtosis (one point per head). Compute Pearson correlation using `scipy.stats.pearsonr`. The paper reports r = 0.84, showing the same heads consistently function as receivers across different reasoning problems.
4. Select the top-$k$ heads by mean kurtosis (paper uses $k = 20$) as the receiver head set. Note which layers these heads concentrate in.

In [ ]:
def replicate_figure_4(
    model,
    tokenizer,
    input_ids: Tensor,
    sentence_boundaries: list[tuple[int, int]],
    proximity_ignore: int = 16,
    top_k: int = 20,
) -> tuple[list[tuple[int, int, float]], np.ndarray]:
    """
    Identify receiver heads via kurtosis of vertical attention scores,
    replicating Figure 4 from the Thought Anchors paper.

    For a single reasoning trace:
    1. Run a forward pass with output_attentions=True to get attention
       matrices for every layer and head
    2. For each head (layer, head_idx), compute sentence-level vertical
       scores: for each sentence s, average the attention weights directed
       toward sentence s from all token positions that are at least
       proximity_ignore positions after sentence s's end boundary
    3. Compute scipy.stats.kurtosis of the per-sentence vertical score
       array for each head, storing results in a (num_layers, num_heads)
       matrix
    4. Identify the top_k heads with highest kurtosis as candidate
       receiver heads
    5. Produce a scatter plot of kurtosis vs layer index to visualize
       outlier heads

    Returns a tuple of:
      - List of (layer, head, kurtosis) for the top_k heads, sorted by
        kurtosis descending
      - The full kurtosis matrix of shape (num_layers, num_heads)
    """
    raise NotImplementedError()

### Receiver head scores by sentence taxonomic category (Figure 5)

The paper's most interpretable whitebox result is that receiver head attention is not uniformly distributed across sentence types. Plan generation and uncertainty management sentences receive significantly higher receiver-head scores than active computation sentences (p < 0.001 for all pairwise comparisons). This directly links the functional role of a sentence (planning, pivoting) to its mechanistic role (being selectively attended to by specialized receiver heads).

The sentence categories used in the paper are: `plan_generation`, `active_computation`, `fact_retrieval`, `uncertainty_management`, `result_consolidation`, `self_checking`, `problem_setup`, and `final_answer_emission`. Each sentence is labeled by an LLM auto-labeler using a structured prompt (`DAG_PROMPT` in `prompts.py` in the paper's repository).

Starting point (single problem):

1. Use an LLM (e.g. via the Anthropic API) with the `DAG_PROMPT` from the paper's `prompts.py` to classify each sentence into function tags based on its role in the reasoning.
2. Compute receiver head scores for each sentence using `compute_receiver_head_scores` (already defined in this notebook).
3. Group sentences by function tag, compute mean receiver-head score per group, and plot a bar chart. Even on a single trace, plan generation and uncertainty management should score noticeably higher than active computation.

Stretch goal - statistical validation across many problems (paper result: p < 0.001):

1. Use the paper's `generate_rec_csvs.py` script to produce CSV files containing receiver head scores and taxonomic labels for every problem in the dataset.
2. For each problem, aggregate receiver head scores by sentence category (per-problem median, to avoid pseudoreplication across sentences within a problem).
3. Run paired t-tests using `scipy.stats.ttest_rel`, comparing pairs of categories using only problems that have examples of both categories. Exclude `final_answer_emission` and `problem_setup` from pairwise comparisons.
4. Plot boxplots stratified by category (as in `plot_rec_taxonomy.py` from the paper code). Compare to Figure 5 of the paper. The result provides statistical support for the core claim: plan generation and uncertainty management sentences are the primary thought anchors - they are the steps that most influence downstream reasoning, both causally (blackbox) and via attention (whitebox).

In [ ]:
def replicate_figure_5(
    model,
    tokenizer,
    sentences: list[str],
    sentence_boundaries: list[tuple[int, int]],
    receiver_heads: list[tuple[int, int]],
    function_tags: list[str],
    categories: list[str] | None = None,
) -> dict[str, list[float]]:
    """
    Compute receiver head scores grouped by sentence taxonomic category,
    replicating Figure 5 from the Thought Anchors paper.

    For a single reasoning trace:
    1. Compute receiver head vertical scores for each sentence using
       compute_receiver_head_scores (already defined in this notebook)
    2. Group sentences by their function tag (from LLM classification)
    3. For each category, collect the receiver head scores of all sentences
       in that category
    4. Plot a bar chart (or boxplot) of mean receiver head score per category
    5. Plan generation and uncertainty management should score noticeably
       higher than active computation

    Returns a dict mapping each category to a list of receiver head scores
    for sentences in that category.
    """
    raise NotImplementedError()

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# Wrap tests.test_compute_suppression_kl so a passing test fires the beacon.
try:
    _dd_orig = tests.test_compute_suppression_kl
    def _dd_wrapped(*args, **kwargs):
        result = _dd_orig(*args, **kwargs)
        _dd_report_complete()
        return result
    tests.test_compute_suppression_kl = _dd_wrapped
except AttributeError:
    print('[Delta Drills] no matching test function — call _dd_report_complete() manually when done.')
